# WDBC Breast Cancer Classification

Notebook format is simple and direct.
Each model has its own section so the results are easy to compare.


## Setup And Data


In [13]:
from pathlib import Path

import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

try:
    import tensorflow as tf
    TF_OK = True
except Exception:
    TF_OK = False

ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

features = ["radius", "texture", "perimeter", "area", "smoothness", "compactness", "concavity", "concave_points", "symmetry", "fractal_dimension"]
columns = ["id", "diagnosis"] + [f"{name}_{part}" for part in ("mean", "se", "worst") for name in features]

df = pd.read_csv(ROOT / "data" / "wdbc.data", names=columns)
print(df.head())

df["target"] = (df["diagnosis"] == "M").astype(int)
print(df["diagnosis"].value_counts())


         id diagnosis  radius_mean  texture_mean  perimeter_mean  area_mean  \
0    842302         M        17.99         10.38          122.80     1001.0   
1    842517         M        20.57         17.77          132.90     1326.0   
2  84300903         M        19.69         21.25          130.00     1203.0   
3  84348301         M        11.42         20.38           77.58      386.1   
4  84358402         M        20.29         14.34          135.10     1297.0   

   smoothness_mean  compactness_mean  concavity_mean  concave_points_mean  \
0          0.11840           0.27760          0.3001              0.14710   
1          0.08474           0.07864          0.0869              0.07017   
2          0.10960           0.15990          0.1974              0.12790   
3          0.14250           0.28390          0.2414              0.10520   
4          0.10030           0.13280          0.1980              0.10430   

   ...  radius_worst  texture_worst  perimeter_worst  area_wor

In [14]:
train, temp = train_test_split(df, test_size=0.4, stratify=df["target"], random_state=42)
valid, test = train_test_split(temp, test_size=0.5, stratify=temp["target"], random_state=42)

x_train_raw = train.drop(columns=["id", "diagnosis", "target"]).values
x_valid_raw = valid.drop(columns=["id", "diagnosis", "target"]).values
x_test_raw = test.drop(columns=["id", "diagnosis", "target"]).values
y_train = train["target"].values
y_valid = valid["target"].values
y_test = test["target"].values

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train_raw)
x_valid = scaler.transform(x_valid_raw)
x_test = scaler.transform(x_test_raw)

summary_rows = []

# Compare models on the validation split only. Keep the test split untouched for the final check.
def show_result(name, y_true, y_pred, y_prob):
    summary_rows.append({
        "model": name,
        "accuracy": round(accuracy_score(y_true, y_pred), 4),
        "precision": round(precision_score(y_true, y_pred), 4),
        "recall": round(recall_score(y_true, y_pred), 4),
        "f1": round(f1_score(y_true, y_pred), 4),
        "roc_auc": round(roc_auc_score(y_true, y_prob), 4),
    })
    print(name)
    print("Validation ROC-AUC:", round(roc_auc_score(y_true, y_prob), 4))
    print(classification_report(y_true, y_pred))


## Distance Based Model

### K-Nearest Neighbours (kNN)


In [15]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(x_train, y_train)
y_pred = knn.predict(x_valid)
y_prob = knn.predict_proba(x_valid)[:, 1]
show_result("kNN", y_valid, y_pred, y_prob)


kNN
Validation ROC-AUC: 0.9959
              precision    recall  f1-score   support

           0       0.93      1.00      0.97        71
           1       1.00      0.88      0.94        43

    accuracy                           0.96       114
   macro avg       0.97      0.94      0.95       114
weighted avg       0.96      0.96      0.96       114



## Probabilistic Model

### Naive Bayes


In [16]:
gnb = GaussianNB()
gnb.fit(x_train, y_train)
y_pred = gnb.predict(x_valid)
y_prob = gnb.predict_proba(x_valid)[:, 1]
show_result("Naive Bayes", y_valid, y_pred, y_prob)


Naive Bayes
Validation ROC-AUC: 0.9912
              precision    recall  f1-score   support

           0       0.92      0.97      0.95        71
           1       0.95      0.86      0.90        43

    accuracy                           0.93       114
   macro avg       0.93      0.92      0.92       114
weighted avg       0.93      0.93      0.93       114



## Linear Model

### Logistic Regression


In [17]:
logreg = LogisticRegression(max_iter=3000)
logreg.fit(x_train, y_train)
y_pred = logreg.predict(x_valid)
y_prob = logreg.predict_proba(x_valid)[:, 1]
show_result("Logistic Regression", y_valid, y_pred, y_prob)


Logistic Regression
Validation ROC-AUC: 1.0
              precision    recall  f1-score   support

           0       0.96      1.00      0.98        71
           1       1.00      0.93      0.96        43

    accuracy                           0.97       114
   macro avg       0.98      0.97      0.97       114
weighted avg       0.97      0.97      0.97       114



/Users/prrat/PHub/Workspace/Projects/mammo/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/prrat/PHub/Workspace/Projects/mammo/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/prrat/PHub/Workspace/Projects/mammo/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/prrat/PHub/Workspace/Projects/mammo/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/prrat/PHub/Workspace/Projects/mammo/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: Runti

## Margin Based Model

### Support Vector Machine (SVM)


In [18]:
svm = SVC(probability=True)
svm.fit(x_train, y_train)
y_pred = svm.predict(x_valid)
y_prob = svm.predict_proba(x_valid)[:, 1]
show_result("SVM", y_valid, y_pred, y_prob)


SVM
Validation ROC-AUC: 0.998
              precision    recall  f1-score   support

           0       0.92      1.00      0.96        71
           1       1.00      0.86      0.93        43

    accuracy                           0.95       114
   macro avg       0.96      0.93      0.94       114
weighted avg       0.95      0.95      0.95       114



## Tree Based Models

### Decision Tree


In [19]:
tree = DecisionTreeClassifier(max_depth=4, random_state=42)
tree.fit(x_train, y_train)
y_pred = tree.predict(x_valid)
y_prob = tree.predict_proba(x_valid)[:, 1]
show_result("Decision Tree", y_valid, y_pred, y_prob)


Decision Tree
Validation ROC-AUC: 0.8942
              precision    recall  f1-score   support

           0       0.92      0.94      0.93        71
           1       0.90      0.86      0.88        43

    accuracy                           0.91       114
   macro avg       0.91      0.90      0.91       114
weighted avg       0.91      0.91      0.91       114



### Random Forest


In [20]:
rf = RandomForestClassifier(n_estimators=300, random_state=42)
rf.fit(x_train, y_train)
y_pred = rf.predict(x_valid)
y_prob = rf.predict_proba(x_valid)[:, 1]
show_result("Random Forest", y_valid, y_pred, y_prob)


Random Forest
Validation ROC-AUC: 0.9949
              precision    recall  f1-score   support

           0       0.95      0.99      0.97        71
           1       0.97      0.91      0.94        43

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114



### Gradient Boosting


In [21]:
gb = GradientBoostingClassifier(random_state=42)
gb.fit(x_train, y_train)
y_pred = gb.predict(x_valid)
y_prob = gb.predict_proba(x_valid)[:, 1]
show_result("Gradient Boosting", y_valid, y_pred, y_prob)


Gradient Boosting
Validation ROC-AUC: 0.9961
              precision    recall  f1-score   support

           0       0.95      0.99      0.97        71
           1       0.97      0.91      0.94        43

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114



## Neural Networks

### sklearn MLPClassifier


In [22]:
mlp = MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=500, random_state=42)
mlp.fit(x_train, y_train)
y_pred = mlp.predict(x_valid)
y_prob = mlp.predict_proba(x_valid)[:, 1]
show_result("MLPClassifier", y_valid, y_pred, y_prob)


/Users/prrat/PHub/Workspace/Projects/mammo/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/prrat/PHub/Workspace/Projects/mammo/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/prrat/PHub/Workspace/Projects/mammo/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


MLPClassifier
Validation ROC-AUC: 1.0
              precision    recall  f1-score   support

           0       0.96      1.00      0.98        71
           1       1.00      0.93      0.96        43

    accuracy                           0.97       114
   macro avg       0.98      0.97      0.97       114
weighted avg       0.97      0.97      0.97       114



/Users/prrat/PHub/Workspace/Projects/mammo/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/prrat/PHub/Workspace/Projects/mammo/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/prrat/PHub/Workspace/Projects/mammo/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/prrat/PHub/Workspace/Projects/mammo/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/prrat/PHub/Workspace/Projects/mammo/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/prrat/PHub/Workspace/Projects/mammo/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered 

### TensorFlow


In [23]:
if TF_OK:
    tf.random.set_seed(42)
    nn = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(x_train.shape[1],)),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ])
    nn.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss="binary_crossentropy", metrics=["accuracy"])
    nn.fit(x_train, y_train, epochs=20, batch_size=32, validation_data=(x_valid, y_valid), verbose=0)
    y_prob = nn.predict(x_valid, verbose=0).ravel()
    y_pred = (y_prob > 0.5).astype(int)
    show_result("TensorFlow NN", y_valid, y_pred, y_prob)
else:
    print("TensorFlow is not installed in this environment yet.")


TensorFlow NN
Validation ROC-AUC: 1.0
              precision    recall  f1-score   support

           0       0.96      1.00      0.98        71
           1       1.00      0.93      0.96        43

    accuracy                           0.97       114
   macro avg       0.98      0.97      0.97       114
weighted avg       0.97      0.97      0.97       114



## Validation Ranking

Keep the full validation leaderboard instead of forcing a single winner. Models are ordered by validation ROC-AUC, then accuracy, then F1.


In [24]:
summary = pd.DataFrame(summary_rows).sort_values(["roc_auc", "accuracy", "f1"], ascending=False).reset_index(drop=True)
top_score = summary.loc[0, "roc_auc"]
top_models = summary.loc[summary["roc_auc"] == top_score, "model"].tolist()

print("Validation ranking (ROC-AUC, accuracy, F1)")
print("Top validation model(s):", ", ".join(top_models))
summary


Validation ranking (ROC-AUC, accuracy, F1)
Top validation model(s): Logistic Regression, MLPClassifier, TensorFlow NN


,model,accuracy,precision,recall,f1,roc_auc
0,Logistic Regression,0.9737,1.0000,0.9302,0.9639,1.0000
1,MLPClassifier,0.9737,1.0000,0.9302,0.9639,1.0000
2,TensorFlow NN,0.9737,1.0000,0.9302,0.9639,1.0000
3,SVM,0.9474,1.0000,0.8605,0.9250,0.9980
4,Gradient Boosting,0.9561,0.9750,0.9070,0.9398,0.9961
5,kNN,0.9561,1.0000,0.8837,0.9383,0.9959
6,Random Forest,0.9561,0.9750,0.9070,0.9398,0.9949
7,Naive Bayes,0.9298,0.9487,0.8605,0.9024,0.9912
8,Decision Tree,0.9123,0.9024,0.8605,0.8810,0.8942
